# IGNORE THIS FILE
# THIS IS NOT MY ASSIGNMENT. ASSIGNMENT IN -> "main.ipynb" file

# institution TOOL

In [1]:
from langchain_community.utilities import SQLDatabase
from pyprojroot import here
import warnings
warnings.filterwarnings("ignore")

C:\Users\HP\AppData\Local\Temp\ipykernel_5580\3992919677.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [11]:
# institution_db
institution_db_path = str(here("22_module_assignment/SQLite_DBs")) + "/institutional.db"
institution_db = SQLDatabase.from_uri(f"sqlite:///{institution_db_path}")

# hospitals_db
hospitals_db_path = str(here("22_module_assignment/SQLite_DBs")) + "/hospitals.db"
hospitals_db = SQLDatabase.from_uri(f"sqlite:///{hospitals_db_path}")

# restaurants_db
restaurants_db_path = str(here("22_module_assignment/SQLite_DBs")) + "/restaurants.db"
restaurants_db = SQLDatabase.from_uri(f"sqlite:///{restaurants_db_path}")

In [12]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
#from langchain_community.chat_models import ChatOllama # optonal

# Load variables from .env
load_dotenv()

llm_provider = "openai" # ollama / openai

In [13]:
# need
if llm_provider == "openai":
    github_token = os.getenv("GITHUB_TOKEN")
    endpoint = "https://models.github.ai/inference"
    model_name = "openai/gpt-4.1-mini"

    if not github_token:
        raise ValueError("GITHUB_TOKEN not set in .env")

    llm = ChatOpenAI(
        model_name=model_name,
        openai_api_key=github_token,
        openai_api_base=endpoint,
        temperature=0.5,
    )

In [14]:
# need
import re
def extract_sql_query_simple(response):
    """
    Extract SQL query from various response formats:
    1. Markdown code blocks: ```sql ... ```
    2. SQLQuery: label format
    3. Plain SQL query
    """
    # First try to extract from markdown code block (```sql ... ```)
    code_block_match = re.search(r'```sql\s*(.*?)\s*```', response, re.IGNORECASE | re.DOTALL)
    if code_block_match:
        sql_query = code_block_match.group(1).strip()
        return sql_query
    
    # Try to find SQL query after "SQLQuery:" label
    label_match = re.search(r"SQLQuery:\s*(.*?)(?:\n|$)", response, re.IGNORECASE)
    if label_match:
        sql_query = label_match.group(1).strip()
        return sql_query
    
    # If response looks like a plain SQL query (starts with SELECT, INSERT, UPDATE, DELETE, etc.)
    plain_sql_match = re.match(r'^\s*(SELECT|INSERT|UPDATE|DELETE|CREATE|ALTER|DROP)\s+', 
                                response, re.IGNORECASE | re.DOTALL)
    if plain_sql_match:
        return response.strip()
    
    return None

# prompts & chain

In [15]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# -----------------------------
# SQL Generation Prompt
# -----------------------------
sql_prompt = ChatPromptTemplate.from_template("""
Given the database schema below, write a SQL query that answers the user's question.

Database Schema:
{schema}

Question: {question}

Return the SQL query in a markdown code block like this:

```sql
YOUR_QUERY_HERE

""")

#-----------------------------
#Final Answer Prompt
#-----------------------------

answer_prompt = PromptTemplate.from_template("""
Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}

Answer:
""")

answer_chain = answer_prompt | llm | StrOutputParser()

# institutions_tool

In [ ]:
from langchain_core.tools import tool
@tool
def institutions_tool(question: str):
    """Answer questions about institutions using the institution database."""
    

    schema_info = institution_db.get_table_info()

    write_query = (
    RunnablePassthrough.assign(schema=lambda _: schema_info)
    | sql_prompt
    | llm
    | StrOutputParser()
    )

    # Generate SQL
    llm_response = write_query.invoke({"question": question})

    # Clean SQL
    sql_query = extract_sql_query_simple(llm_response)

    if sql_query is None:
        return "Failed to generate SQL query."

    # Execute SQL
    sql_result = institution_db._execute(sql_query)

    # Generate final answer
    final_answer = answer_chain.invoke({
    "question": question,
    "query": sql_query,
    "result": sql_result
    })

    return final_answer

'\nprint(\n    institutions_tool(\n        "count the number of institution inside BARISAL"\n    )\n)\n'

# hospitals_tool

In [ ]:
@tool
def hospitals_tool(question: str):
    """Answer questions about hospitals using the hospitals database."""
    
    schema_info = hospitals_db.get_table_info()

    write_query = (
    RunnablePassthrough.assign(schema=lambda _: schema_info)
    | sql_prompt
    | llm
    | StrOutputParser()
    )

    # Generate SQL
    llm_response = write_query.invoke({"question": question})

    # Clean SQL
    sql_query = extract_sql_query_simple(llm_response)

    if sql_query is None:
        return "Failed to generate SQL query."

    # Execute SQL
    sql_result = hospitals_db._execute(sql_query)

    # Generate final answer
    final_answer = answer_chain.invoke({
    "question": question,
    "query": sql_query,
    "result": sql_result
    })

    return final_answer

'\nprint(\n    hospitals_tool(\n        "List top 10 hospitals in Dhaka with bed capacity, show name and bed capacity"\n    )\n)\n'

# restaurants tool

In [ ]:
@tool
def restaurants_tool(question: str): 
    """Answer questions about restaurant using the restaurants database."""
    
    schema_info = restaurants_db.get_table_info()

    write_query = (
    RunnablePassthrough.assign(schema=lambda _: schema_info)
    | sql_prompt
    | llm
    | StrOutputParser()
    )

    # Generate SQL
    llm_response = write_query.invoke({"question": question})

    # Clean SQL
    sql_query = extract_sql_query_simple(llm_response)

    if sql_query is None:
        return "Failed to generate SQL query."

    # Execute SQL
    sql_result = restaurants_db._execute(sql_query)

    # Generate final answer
    final_answer = answer_chain.invoke({
    "question": question,
    "query": sql_query,
    "result": sql_result
    })

    return final_answer

'\nprint(\n    restaurants_tool(\n        "give me the 3rd resturont name from the database."\n    )\n)\n'

# search tool

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Make sure TAVILY_API_KEY is set in your environment
tavily_search = TavilySearchResults(max_results=5)

@tool
def web_search_tool(question: str):
    """Answer general knowledge questions using web search (e.g. policies, current events, facts not in our databases)."""

    results = tavily_search.invoke({"query": question})

    # results is a list of dicts: [{"url":..., "content":...}, ...]
    context = "\n\n".join(
        f"Source: {r.get('url')}\nContent: {r.get('content')}"
        for r in results
    )

    final_answer = web_answer_chain.invoke({
        "question": question,
        "context": context
    })

    return final_answer

# MAIN AGENT

In [ ]:
from langchain.agents import create_agent

# Main Agent
main_agent = create_agent(
    model=llm,
    tools=[
        institutions_tool,
        hospitals_tool,
        restaurants_tool,
    ]
)

There are 9,876 hospitals in Dhaka.


# input Q

In [34]:
response = main_agent.invoke({
    "messages": [
        ("user", "How many hospitals are there in Dhaka?")
    ]
})

print(response["messages"][-1].content)

There are 9,876 hospitals in Dhaka.
